In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers

In [ ]:
import torch
import math
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorWithPadding, set_seed
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm
import re

In [ ]:
SEED = 42
BATCH_SIZE = 8
MAX_LENGTH = 512
STRIDE = 128
MLM_PROBABILITY = 0.15

In [ ]:
MODELS = [
    "dbmdz/bert-base-italian-xxl-cased",
    "linhd-postdata/alberti-bert-base-multilingual-cased",
    "mattiaferrarini/saba"
]
DATASET_PATH = "/content/drive/My Drive/italian_poems/italian_poems_test_rhymes.json"

In [ ]:
def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

fix_seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

In [ ]:
for model_id in MODELS:
    print(f"\nEvaluating Attention: {model_id}")
    fix_seed(SEED)

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if not tokenizer.is_fast:
        print(f"Warning: {model_id} tokenizer is not 'fast'. Loading fast version.")
        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    model = AutoModelForMaskedLM.from_pretrained(model_id, output_attentions=True).to(device)
    model.eval()

    # Filter only by length
    def filter_long_samples(examples):
        tokenized = tokenizer(examples["text"], truncation=False, padding=False)
        return [len(ids) <= MAX_LENGTH for ids in tokenized["input_ids"]]

    filtered_dataset = dataset.filter(
        filter_long_samples,
        batched=True,
        desc=f"Filtering > {MAX_LENGTH} tokens"
    )
    print("Total poems for evaluation:", len(filtered_dataset))

    #  Tokenization, masking and attention target tagging
    def process_data(examples):
        tokenized = tokenizer(
            examples["text"],
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
            return_offsets_mapping=True,
            return_special_tokens_mask=True
        )

        new_input_ids = []
        new_labels = []
        mask_group_ids = []
        target_all_mask = []
        target_rhyme_mask = []
        is_rhyming_sample = []

        for i, text in enumerate(examples["text"]):
            raw_tags = examples["rhyme_tags"][i]
            input_ids = tokenized["input_ids"][i]
            offsets = tokenized["offset_mapping"][i]
            special_mask = tokenized["special_tokens_mask"][i]

            labels = [-100] * len(input_ids)
            current_mask_group = [0] * len(input_ids)
            current_target_all = [0] * len(input_ids)
            current_target_rhyme = [0] * len(input_ids)

            # Robust line extraction
            line_spans = []
            last_pos = 0
            for match in re.finditer(r'\n', text):
                line_spans.append((last_pos, match.start()))
                last_pos = match.end()
            if last_pos < len(text):
                line_spans.append((last_pos, len(text)))

            # Filter empty lines 
            valid_lines = []
            for start, end in line_spans:
                line_content = text[start:end]
                if line_content.strip(): # Only keep if not empty
                    valid_lines.append((start, end))

            # Determine target tag
            target_tag = None
            has_rhyme = 0

            # We need valid lines to align with raw_tags
            if len(valid_lines) > 0 and len(raw_tags) == len(valid_lines):
                target_tag = raw_tags[-1]
                # Check if this tag appeared previously
                if target_tag is not None and raw_tags.count(target_tag) >= 2:
                    has_rhyme = 1

            is_rhyming_sample.append(has_rhyme)

            # Process previous lines
            for j, (start_char, end_char) in enumerate(valid_lines[:-1]):
                line_text = text[start_char:end_char]
                current_line_tag = raw_tags[j]

                # Find last word
                matches = list(re.finditer(r"\b\w+\b", line_text))
                if matches:
                    last_match = matches[-1]
                    word_global_start = start_char + last_match.start()
                    word_global_end = start_char + last_match.end()

                    # Map to tokens
                    for idx, (tok_start, tok_end) in enumerate(offsets):
                        if special_mask[idx] or tok_start == tok_end: continue

                        if tok_start < word_global_end and tok_end > word_global_start:
                            current_target_all[idx] = 1
                            if target_tag is not None and current_line_tag == target_tag:
                                current_target_rhyme[idx] = 1

            # Process last line
            if len(valid_lines) > 0:
                start_char, end_char = valid_lines[-1]
                last_line_text = text[start_char:end_char]

                matches = list(re.finditer(r"\b\w+\b", last_line_text))
                masked_at_least_one = False

                if matches:
                    last_match = matches[-1]
                    word_global_start = start_char + last_match.start()
                    word_global_end = start_char + last_match.end()

                    for idx, (tok_start, tok_end) in enumerate(offsets):
                        if special_mask[idx] or tok_start == tok_end: continue

                        if tok_start < word_global_end and tok_end > word_global_start:
                            labels[idx] = input_ids[idx]
                            input_ids[idx] = tokenizer.mask_token_id
                            current_mask_group[idx] = 1
                            masked_at_least_one = True

                # Fallback
                if not masked_at_least_one:
                    for idx in range(len(input_ids)-1, -1, -1):
                        if input_ids[idx] != tokenizer.pad_token_id and not special_mask[idx]:
                            labels[idx] = input_ids[idx]
                            input_ids[idx] = tokenizer.mask_token_id
                            current_mask_group[idx] = 1
                            break

            new_input_ids.append(input_ids)
            new_labels.append(labels)
            mask_group_ids.append(current_mask_group)
            target_all_mask.append(current_target_all)
            target_rhyme_mask.append(current_target_rhyme)

        tokenized["input_ids"] = new_input_ids
        tokenized["labels"] = new_labels
        tokenized["mask_group_ids"] = mask_group_ids
        tokenized["target_all_mask"] = target_all_mask
        tokenized["target_rhyme_mask"] = target_rhyme_mask
        tokenized["is_rhyming"] = is_rhyming_sample
        return tokenized

    tokenized_datasets = filtered_dataset.map(
        process_data,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Processing"
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    dataloader = DataLoader(
        tokenized_datasets,
        batch_size=BATCH_SIZE,
        collate_fn=data_collator
    )

    # Metrics containers
    stats = {
        "all_lines":   {"total_attn": 0.0, "count": 0},
        "rhyme_lines": {"total_attn": 0.0, "count": 0}
    }

    for batch in tqdm(dataloader, desc="Eval Attention"):
        # Move custom masks to GPU
        mask_groups = batch.pop("mask_group_ids").to(device)       # [B, Seq]
        target_all = batch.pop("target_all_mask").to(device)       # [B, Seq]
        target_rhyme = batch.pop("target_rhyme_mask").to(device)   # [B, Seq]
        is_rhyming = batch.pop("is_rhyming").to(device)            # [B]

        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            outputs = model(**batch)
            # We take the last layer attentions
            last_layer_attn = outputs.attentions[-1]
            avg_attn = last_layer_attn.mean(dim=1)

        batch_size, seq_len = mask_groups.shape

        for b in range(batch_size):
            # Identify indices of the [MASK] tokens
            source_indices = (mask_groups[b] == 1).nonzero(as_tuple=True)[0]

            if len(source_indices) == 0: continue

            # Extract attention rows corresponding to the mask tokens
            mask_attn_rows = avg_attn[b, source_indices, :]
            focus_vector = mask_attn_rows.mean(dim=0)

            # Calculate mass on all previous final words
            all_target_indices = (target_all[b] == 1)
            mass_all = focus_vector[all_target_indices].sum().item()

            # Only count if there were actual targets to look at
            if all_target_indices.sum() > 0:
                stats["all_lines"]["total_attn"] += mass_all
                stats["all_lines"]["count"] += 1

            # Calculate mass on rhyming final wordsif is_rhyming[b] == 1:
                rhyme_target_indices = (target_rhyme[b] == 1)
                mass_rhyme = focus_vector[rhyme_target_indices].sum().item()

                # Only count if there were previous rhymes to look at
                if rhyme_target_indices.sum() > 0:
                    stats["rhyme_lines"]["total_attn"] += mass_rhyme
                    stats["rhyme_lines"]["count"] += 1

    # Reporting
    print(f"\nAttention Results for {model_id}:")
    print("=" * 60)

    # All endings
    s_all = stats["all_lines"]
    if s_all["count"] > 0:
        avg_all = s_all["total_attn"] / s_all["count"]
        print(f"Avg Attention to ALL previous line endings:   {avg_all:.4f} (over {s_all['count']} samples)")
    else:
        print("Avg Attention to ALL previous line endings:   N/A")

    # Rhymes
    s_rhyme = stats["rhyme_lines"]
    if s_rhyme["count"] > 0:
        avg_rhyme = s_rhyme["total_attn"] / s_rhyme["count"]
        print(f"Avg Attention to RHYMING previous endings:    {avg_rhyme:.4f} (over {s_rhyme['count']} samples)")
    else:
        print("Avg Attention to RHYMING previous endings:    N/A")

    print("-" * 60)

    del model
    torch.cuda.empty_cache()